# DentVLM panoramic pipeline (Kaggle runner)

One image goes through the paper's own protocol; every question is one DentVLM was trained on:

```text
panoramic X-ray
   -> 13 yes/no questions, one per DentVLM panoramic task (Supplementary Table 7 wording), whole image
   -> line 1 of each reply = Yes/No; the rationale names the location with the nine trained
      descriptors -> six dental-arch cells; multiplicity = number of cells named
   -> optional knobs: phrasings=3 vote, count_question for an out-of-distribution count, location="crops"
      (every cell asked every task, whatever the whole image answered; the whole-image answers are kept
      separately)
   -> JSON per image, deterministic dentist summary, TP/FP/TN/FN evaluation per finding and per cell
   -> location truth: ground-truth boxes are translated into the same six cells by a vision LLM
      (numbered boxes drawn on the image -> FDI quadrant x anterior/posterior), by DentVLM itself
      (experimental), or by fixed windows
   -> dentist report: a text LLM (the reporter role) gets the findings of one image as one dense JSON (every
      finding and extra task with its question and answer, every region, multiplicity, explicit statuses) and
      returns a classified report in the dentist's language (report_language); verified against the data,
      corrected once if needed, rendered to Markdown
```

**CELL 3 is the only cell to edit.** It holds one dictionary per configuration: a name plus the knobs that
configuration changes - local or hosted backend, the analyzer / adapter / reporter models, the protocol knobs,
the location truth. Everything a configuration does not mention comes from `experiments.DEFAULTS`. Every
experiment answers the same images, writes into `<output_root>/<name>/`, and CELL 11 scores them side by side
and ranks them, so one session says which configuration works best instead of one configuration per session.
Run the cells in order.

In [ ]:
# ============================================================
# CELL 1 - Python dependencies (llama.cpp is compiled later with CUDA)
# ============================================================
%pip install -q "huggingface_hub>=0.26" "openai>=1.55" "requests>=2.31" "pillow>=10.0" "pandas>=1.5"
print("Python dependencies installed.")

In [ ]:
# ============================================================
# CELL 2 - Locate and import the project files
# ============================================================
import os, sys, json, shutil, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/dental_x-ray")  # folder holding the project .py files
REQUIRED_PROJECT_FILES = {"dental_pipeline.py", "dental_eval.py", "llama_runtime.py", "location_adapter.py",
                          "llm_api.py", "report_writer.py", "dental_analysis.py", "experiments.py"}
if not PROJECT_DIR.is_dir() and Path.cwd().joinpath("dental_pipeline.py").is_file():
    PROJECT_DIR = Path.cwd()
missing = [f for f in REQUIRED_PROJECT_FILES if not (PROJECT_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing project files in {PROJECT_DIR}: {missing}")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import dental_pipeline as dp
import dental_eval as ev
import dental_analysis as da
import location_adapter as la
import llm_api
import experiments as xp
import report_writer as rw
from llama_runtime import LlamaCppServer, build_llama_cpp, convert_to_gguf, download_gguf, local_gguf
print("PROJECT_DIR =", PROJECT_DIR)
print("Project imports succeeded.")

In [ ]:
# ============================================================
# CELL 3 - CONFIGURATION: the experiments to run and compare (the only cell to edit)
# ============================================================
OUTPUT_ROOT = "/kaggle/working/dentvlm_local_compact_diagnostics_v1"

# Provider endpoints and credentials: define each provider once. Keys are read from environment variables or
# Kaggle Secrets (Add-ons > Secrets); missing keys are allowed until that provider is assigned to a role.
PROVIDERS = {
    "openai": {"base_url": "https://api.openai.com/v1",
               "api_key": llm_api.secret("OPENAI_API_KEY", required=False)},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",
                   "api_key": llm_api.secret("OPENROUTER_API_KEY", required=False)},
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1",
               "api_key": llm_api.secret("NVIDIA_API_KEY", required=False)},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "api_key": llm_api.secret("GEMINI_API_KEY", required=False)},
}
llm_api.configure_providers(PROVIDERS)

# What every experiment starts from. Any knob of experiments.DEFAULTS may be set here (print(xp.DEFAULTS),
# or read experiments.py, for the full list with its comments): the three model roles, the protocol knobs,
# the location truth, the report language, the local llama.cpp settings.
SHARED = {
    "output_root": OUTPUT_ROOT,
    "backend": "local",
    "phrasings": 1,
    "region_vote": "union",
    "location": "rationale",
    "count_question": False,
    "ask_untrained": False,
    "extra_tasks": True,
    "parse_retries": 1,
    "evaluate_location": True,
    "location_truth": "geometry",
    "temperature": 0.0,
    "max_tokens": 512,
    "ctx_size": 16384,
    "image_max_tokens": 8192,
    "image_min_tokens": None,
    "cache_prompt": True,
    "reuse_local_responses": True,
    # The analyzer answers the task questions (used when "backend" is "api").
    "analyzer": {"provider": "openrouter", "model": "qwen/qwen3-vl-235b-a22b-thinking",
                 "request_options": {"extra_body": {"provider": {"only": ["novita"],
                                                                 "allow_fallbacks": False}}}},
    # The adapter translates ground-truth boxes into the six cells (location_truth="llm").
    "adapter": {"provider": "openai", "model": "gpt-5",
                "token_param": "max_completion_tokens", "temperature": None,
                "max_output_tokens": 8192, "max_boxes_per_call": 12},
    # The reporter turns one image's findings into a dentist report (CELL 14). It never sees the image.
    "reporter": {"provider": "openai", "model": "gpt-5",
                 "token_param": "max_completion_tokens", "temperature": None, "max_output_tokens": 8192,
                 "include_rationale": False},
    "report_language": "English",  # every sentence of the report, e.g. "Persian", "German"
}

# One dictionary per configuration: a name, plus only the knobs that configuration changes. Every experiment
# answers the same images and writes into <output_root>/<name>/; CELL 11 ranks them. Identical local model
# requests are reused from <output_root>/_response_cache and remain visible in each experiment. A dictionary knob merges
# key by key, so {"analyzer": {"model": ...}} keeps the provider and request options set above.
EXPERIMENTS = xp.build([
    {"name": "01-paper-rationale"},
    {"name": "02-three-phrasings", "phrasings": 3},
    {"name": "03-with-counts", "count_question": True},             # explicitly out of distribution
    {"name": "04-six-cell-crops", "location": "crops"},          # comparison only; crops are out of distribution
    {"name": "05-untrained-classes", "ask_untrained": True},         # five unsupported UMFIH classes
    {"name": "06-image-budget-1369", "image_max_tokens": 1369},      # authors' 1024x1024 ablation
], shared=SHARED)
xp.show(EXPERIMENTS)

# Datasets to run and score; the same images for every experiment, so the comparison is paired.
# kind "yolo": UMFIH 14-class layout; kind "dentex": DENTEX split. "limit": first N images (sorted by id).
DATASETS = [
    {"name": "umfih_test", "kind": "yolo", "limit": None,
     "images": "/kaggle/working/umfih_14class/data/test/images",
     "labels": "/kaggle/working/umfih_14class/data/test/labels"},
    # {"name": "umfih_external", "kind": "yolo", "limit": None,
    #  "images": "/kaggle/working/umfih_14class/external/images",
    #  "labels": "/kaggle/working/umfih_14class/external/labels"},
    # DENTEX: fully labeled train split (705 images, COCO-style JSON) and/or the 50-image validation split
    # (validation_triple.json). DentVLM's authors used only the 242-image official test split for their
    # external validation, so both of these are held-out for the model as well.
    # {"name": "dentex_train", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"},
    # {"name": "dentex_val", "kind": "dentex", "limit": None,
    #  "images": "/kaggle/working/DENTEX/validation_data/quadrant_enumeration_disease/xrays",
    #  "annotations": "/kaggle/working/DENTEX/validation_triple.json"},
]

# Machine settings for the local backend: where llama.cpp is built and served, and where the GGUF files are
# converted and cached. The same for every local experiment; what the model is and sees (checkpoint, source,
# context, image tokens) is a knob of the experiment instead.
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST, SERVER_PORT, SERVER_ALIAS = "127.0.0.1", 8080, "dentvlm"
SERVER_LOG_PATH = "/kaggle/working/llama_dentvlm_server.log"
SERVER_STARTUP_TIMEOUT = 300.0
MODEL_DIR = "/kaggle/working/models/dentvlm"
CONVERT_WORK_DIR = "/tmp/dentvlm_hf"  # scratch for the 17 GB safetensors; not persisted
HF_TOKEN_SECRET = "HF_TOKEN"          # Kaggle secret holding a Hugging Face token (or set env HF_TOKEN)
CUDA_ARCH = None                      # None = auto-detect (P100 fallback 60)
BUILD_JOBS = 4

LOCAL = [c for c in EXPERIMENTS if xp.is_local(c)]
print(f"\ndatasets = {[d['name'] for d in DATASETS]} | local experiments = {[c['name'] for c in LOCAL]}")

In [ ]:
# ============================================================
# CELL 4 - Environment diagnostics
# ============================================================
import platform
print("Python:", platform.python_version(), "|", platform.platform())


def detect_cuda_arch(fallback="60"):
    try:
        output = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                         text=True, stderr=subprocess.STDOUT)
        arch = output.strip().splitlines()[0].strip().replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)
    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = None
if LOCAL:
    for executable in ("git", "cmake", "nvcc", "nvidia-smi"):
        print(f"{executable:12s}:", shutil.which(executable))
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    for executable in ("cmake", "git", "nvcc"):
        if not shutil.which(executable):
            raise RuntimeError(f"{executable} is required to build llama.cpp; enable a GPU accelerator.")
    CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
    print("CUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)
else:
    print("No local experiment; llama.cpp checks skipped.")

In [ ]:
# ============================================================
# CELL 5 - Build or find the pinned llama.cpp server
# ============================================================
LLAMA_SERVER = None
if LOCAL:
    # Reuses an existing build only if it was built from LLAMA_CPP_REF; otherwise rebuilds.
    LLAMA_SERVER = Path(build_llama_cpp(source_dir=LLAMA_CPP_DIR, cuda_arch=CUDA_ARCH_RESOLVED,
                                        jobs=BUILD_JOBS, ref=LLAMA_CPP_REF)).resolve()
    print("llama-server =", LLAMA_SERVER)
else:
    print("No local experiment; llama.cpp build skipped.")

In [ ]:
# ============================================================
# CELL 6 - Local DentVLM: the GGUF files, and one llama.cpp server at a time
# ============================================================
# DentVLM (Hugging Face ZJU-AI4H/DentVLM, gated with automatic approval, CC BY-NC 4.0) ships as bf16
# safetensors; model_source "convert" downloads and converts it once (~17 GB scratch, ~9.5 GB kept),
# "local" uses files already in MODEL_DIR (an attached Kaggle dataset), "hf" downloads them from your own
# gguf_repo_id. Experiments asking for the same files convert or download once; experiments asking for the
# same server settings share the running server.
import requests

MODELS = {}
for cfg in LOCAL:
    key = xp.model_key(cfg)
    if key in MODELS:
        continue
    source, repo_id, model_filename, mmproj_filename = key
    hf_token = llm_api.secret(HF_TOKEN_SECRET, required=False)
    if source == "convert":
        if not hf_token:
            print(f"No Hugging Face token under {HF_TOKEN_SECRET!r}; only model_source 'local' works without one.")
        MODELS[key] = convert_to_gguf(MODEL_DIR, LLAMA_CPP_DIR, hf_token=hf_token, work_dir=CONVERT_WORK_DIR,
                                      model_filename=model_filename, mmproj_filename=mmproj_filename)
        print("Keep the two files above (private Kaggle dataset or Hugging Face repo) and switch "
              "model_source to 'local' or 'hf' for the next session.")
    elif source == "local":
        MODELS[key] = local_gguf(MODEL_DIR, model_filename, mmproj_filename)
    else:
        MODELS[key] = download_gguf(MODEL_DIR, repo_id, model_filename, mmproj_filename, hf_token)
    for label, path in (("Language model", MODELS[key].model_path), ("Vision projector", MODELS[key].mmproj_path)):
        print(f"{label}: {path} ({Path(path).stat().st_size / 1024**3:.2f} GiB)")

SERVER = SERVER_SETTINGS = None


def local_server(cfg):
    """The llama.cpp server for one experiment, restarted only when its local settings change."""
    global SERVER, SERVER_SETTINGS
    if SERVER is not None and SERVER_SETTINGS == xp.server_key(cfg):
        return SERVER
    if SERVER is not None:
        SERVER.stop()
    files = MODELS[xp.model_key(cfg)]
    SERVER = LlamaCppServer(binary=LLAMA_SERVER, model_path=files.model_path, mmproj_path=files.mmproj_path,
                            host=SERVER_HOST, port=SERVER_PORT, alias=SERVER_ALIAS,
                            n_gpu_layers=cfg["n_gpu_layers"], ctx_size=cfg["ctx_size"],
                            image_max_tokens=cfg["image_max_tokens"], image_min_tokens=cfg["image_min_tokens"],
                            startup_timeout=SERVER_STARTUP_TIMEOUT, log_path=SERVER_LOG_PATH)
    SERVER.start(reuse_existing=False)
    ids = [m.get("id") for m in requests.get(f"{SERVER.base_url}/v1/models", timeout=10).json().get("data", [])]
    if SERVER_ALIAS not in ids:
        raise RuntimeError(f"Expected alias {SERVER_ALIAS!r}; /v1/models returned {ids}. See {SERVER_LOG_PATH}.")
    SERVER_SETTINGS = xp.server_key(cfg)
    print(f"{cfg['name']}: DentVLM server verified at {SERVER.base_url}")
    return SERVER


def open_runner(cfg):
    """The analyzer of one experiment: its hosted model, or the local server started above."""
    return xp.runner(cfg, local_server(cfg) if xp.is_local(cfg) else None)


print("Model files ready:", len(MODELS), "| local server helper defined")

In [ ]:
# ============================================================
# CELL 7 - Load ground truth for every dataset (no model calls)
# ============================================================
GT = {}
for spec in DATASETS:
    if spec["kind"] == "yolo":
        gt = ev.load_yolo(spec["images"], spec["labels"])
    elif spec["kind"] == "dentex":
        gt = ev.load_dentex(spec["images"], spec["annotations"])
    else:
        raise ValueError(f"unknown dataset kind {spec['kind']!r}")
    if spec.get("limit"):
        gt = dict(sorted(gt.items())[: spec["limit"]])
    GT[spec["name"]] = gt
    positives = sum(1 for g in gt.values() if g["boxes"])
    print(f"{spec['name']}: {len(gt)} images, {positives} with at least one finding, "
          f"{sum(len(g['boxes']) for g in gt.values())} boxes")

In [ ]:
# ============================================================
# CELL 8 - Smoke test: raw replies of every experiment on a few images
# ============================================================
# Sends the primary implant and caries questions to the first images ("smoke_images" in CELL 3; 0 skips it).
# Expect line 1 to be Yes/No on every reply, a location descriptor on most Yes replies, and no truncation.
SMOKE = {}
for cfg in [c for c in EXPERIMENTS if c["smoke_images"]]:
    runner = open_runner(cfg)
    paths = [g["path"] for _, g in sorted(next(iter(GT.values())).items())[: cfg["smoke_images"]]]
    rows = []
    for path in paths:
        for task in ("implant", "caries"):
            reply = runner.ask(path, dp.questions_for(task)[0])
            rows.append({"experiment": cfg["name"], "image": Path(path).name, "task": task,
                         "answer": dp.extract_answer(reply["text"]), "regions": dp.extract_regions(reply["text"]),
                         "truncated": reply["truncated"], "text": reply["text"]})
            print(f"--- {cfg['name']} | {rows[-1]['image']} | {task} | answer={rows[-1]['answer']} | "
                  f"regions={rows[-1]['regions']} | truncated={rows[-1]['truncated']}\n{reply['text'][:600]}\n")
    SMOKE[cfg["name"]] = rows
    yes_rows = [r for r in rows if r["answer"] == "yes"]
    print(f"{cfg['name']}: parsed line-1 answers {sum(r['answer'] is not None for r in rows)}/{len(rows)} | "
          f"yes replies naming a region {sum(bool(r['regions']) for r in yes_rows)}/{len(yes_rows)} | "
          f"truncated {sum(r['truncated'] for r in rows)}\n")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_ROOT, "smoke.json").write_text(json.dumps(SMOKE, indent=1, ensure_ascii=False), encoding="utf-8")

In [ ]:
# ============================================================
# CELL 9 - Run every experiment (resumable: finished images are skipped)
# ============================================================
# One experiment at a time, one directory each: <output_root>/<name>/<dataset>/. The resolved configuration
# is saved as experiment.json and hashed into the run manifest, so a changed knob can never be mixed into a
# resumed run. An experiment that fails is reported and the sweep continues with the next one.
import traceback

FAILED = {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    print(f"\n{'=' * 78}\n=== {name}: {xp.analyzer_name(cfg)} | {xp.protocol(cfg)}\n{'=' * 78}")
    try:
        xp.record(cfg)
        runner = open_runner(cfg)
        print(f"  runner = {runner.settings()}")
        for spec in DATASETS:
            images = {image_id: g["path"] for image_id, g in GT[spec["name"]].items()}
            out = dp.run_dataset(runner, images, xp.run_dir(cfg, spec["name"]), protocol=xp.protocol(cfg),
                                 resume=True, provenance=xp.provenance(cfg, llama_cpp_ref=LLAMA_CPP_REF))
            print("  saved:", out)
    except Exception:
        FAILED[name] = traceback.format_exc()
        print(f"!! {name} FAILED; the other experiments continue\n{FAILED[name]}")

print("\nfinished:", [c["name"] for c in EXPERIMENTS if c["name"] not in FAILED], "| failed:", sorted(FAILED))

In [ ]:
# ============================================================
# CELL 10 - Location truth: translate ground-truth boxes into DentVLM's six cells (resumable)
# ============================================================
# Independent of the model run, and keyed by the adapter rather than by the experiment: every experiment
# using the same adapter reads the same translated boxes from
# <output_root>/location_truth/<dataset>/<adapter>/ instead of paying for them again.
# One JSON per image under boxes/, drawn images under drawn/ for audit; unparseable replies follow
# location_failure_policy, and every attempt and fallback is saved.
ADAPTED, TRUTH_DIRS = {}, {}
for cfg in EXPERIMENTS:
    name = cfg["name"]
    for spec in DATASETS:
        dataset = spec["name"]
        if not cfg["evaluate_location"] or cfg["location_truth"] == "geometry":
            print(f"{name}/{dataset}: fixed windows (and FDI tooth numbers where the dataset has them)")
            continue
        out = xp.truth_dir(cfg, dataset)
        if out in TRUTH_DIRS:
            ADAPTED[name, dataset] = TRUTH_DIRS[out]
            print(f"{name}/{dataset}: reuses {out}")
            continue
        adapter = xp.location_adapter(cfg, open_runner(cfg) if cfg["location_truth"] == "fdm" else None)
        TRUTH_DIRS[out] = ADAPTED[name, dataset] = la.adapt_dataset(adapter, GT[dataset], out, resume=True)
        print(f"{name}/{dataset}: {la.summarize(ADAPTED[name, dataset])}")
        agreement = ev.truth_agreement(GT[dataset], ADAPTED[name, dataset])
        if agreement["boxes_with_fdi"]:
            # DENTEX carries FDI tooth numbers: exact cells, so this is the adapter's own accuracy.
            print(f"{name}/{dataset}: adapter vs FDI truth {agreement}")

In [ ]:
# ============================================================
# CELL 11 - Evaluate and compare every experiment
# ============================================================
import pandas as pd
from IPython.display import display

# Each experiment is scored against its own location truth and written to <name>/<dataset>/evaluation/.
REPORTS = {}
for cfg in EXPERIMENTS:
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        results = dp.load_results(xp.run_dir(cfg, dataset))
        if not (set(GT[dataset]) & set(results)):
            print(f"{name}/{dataset}: no results under {xp.run_dir(cfg, dataset)}; run CELL 9 first.")
            continue
        truth = ev.apply_adapted(GT[dataset], ADAPTED[name, dataset]) if (name, dataset) in ADAPTED else GT[dataset]
        REPORTS[name, dataset] = ev.evaluate(truth, results, dataset=dataset,
                                             out_dir=xp.run_dir(cfg, dataset) / "evaluation",
                                             evaluate_location=cfg["evaluate_location"])

# One joined artifact supplies the notebook and reusable CSVs. Full rows retain image IDs; the notebook
# shows compact decision tables so raw confusion counts, denominators, and protocol changes stay together.
VIEWS = da.compact_views(GT, REPORTS)
LEADERBOARD = pd.DataFrame(VIEWS["experiment_overview"])
FINDING_COMPARISON = pd.DataFrame(VIEWS["finding_comparison"])
SITUATION_COMPARISON = pd.DataFrame(VIEWS["situation_comparison"])
SITUATION_FINDING_COMPARISON = pd.DataFrame(VIEWS["situation_finding_comparison"])
STAGE_COMPARISON = pd.DataFrame(VIEWS["stage_comparison"])
PHRASING_COMPARISON = pd.DataFrame(VIEWS["phrasing_comparison"])
VOTE_REPLAY_COMPARISON = pd.DataFrame(VIEWS["vote_replay_comparison"])
PARSE_RECOVERY_COMPARISON = pd.DataFrame(VIEWS["parse_recovery_comparison"])
CALL_USAGE_COMPARISON = pd.DataFrame(VIEWS["call_usage_comparison"])

if not LEADERBOARD.empty:
    Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
    LEADERBOARD = LEADERBOARD.sort_values(["dataset", "f1"], ascending=[True, False])
    LEADERBOARD.to_csv(Path(OUTPUT_ROOT) / "leaderboard.csv", index=False)
    ev.write_report(VIEWS, Path(OUTPUT_ROOT) / "overview")
    protocol = ["dataset", "experiment", "phrasings", "region_vote", "location",
                "count_question", "ask_untrained", "evaluate_location"]
    print("===== finding detection (rates use scored checks only) =====")
    display(LEADERBOARD[protocol + ["images", "annotated_checks", "not_assessed_checks",
        "expected_checks", "scored_checks", "TP", "TN", "FP", "FN",
        "excluded_unparseable_checks", "coverage", "sensitivity", "specificity", "ppv", "f1",
        "macro_f1", "mean_false_alarms_per_image", "logical_calls", "inference_calls", "cache_hits",
        "mean_calls_per_image", "mean_inference_calls_per_image", "cache_hit_rate"]].set_index(protocol))

    count_rows = LEADERBOARD[LEADERBOARD["count_question"] == True]
    if not count_rows.empty:
        print("===== optional count question (out-of-distribution for DentVLM) =====")
        display(count_rows[["dataset", "experiment", "count_n_scored", "count_exact_rate",
            "count_within_1_rate", "count_mae", "count_strict_n", "count_strict_mae",
            "count_unparseable"]].set_index(["dataset", "experiment"]))

    location_rows = LEADERBOARD[(LEADERBOARD["evaluate_location"] == True) &
                                (LEADERBOARD["location"] != "none")]
    if not location_rows.empty:
        print("===== DentVLM six-cell location =====")
        display(location_rows[["dataset", "experiment", "location", "localized_cases",
            "location_TP", "location_TN", "location_FP", "location_FN", "location_f1",
            "region_exact_rate", "region_jaccard", "region_presence_TP", "region_presence_TN",
            "region_presence_FP", "region_presence_FN", "region_presence_f1",
            "side_agreement_rate", "location_excluded"]].set_index(["dataset", "experiment"]))

if not FINDING_COMPARISON.empty:
    assessed = FINDING_COMPARISON[FINDING_COMPARISON["assessment_status"] != "not_assessed"]
    print("===== each assessed finding =====")
    display(assessed[["dataset", "experiment", "condition", "trained_task", "annotated_images",
        "annotated_positives", "scored_checks", "TP", "TN", "FP", "FN", "unparseable",
        "coverage", "sensitivity", "specificity", "ppv", "f1"]].set_index(
        ["dataset", "experiment", "condition"]))
    unassessed = FINDING_COMPARISON[FINDING_COMPARISON["assessment_status"] == "not_assessed"]
    if not unassessed.empty:
        print("===== annotated findings not assessed by this DentVLM protocol =====")
        display(unassessed[["dataset", "experiment", "condition", "trained_task",
            "annotated_images", "annotated_positives", "not_assessed_checks"]].set_index(
            ["dataset", "experiment", "condition"]))

if not SITUATION_COMPARISON.empty:
    print("===== case situations (descriptive slices) =====")
    display(SITUATION_COMPARISON[["dataset", "experiment", "situation", "group", "images_scored",
        "not_assessed_checks", "expected_finding_checks", "scored_finding_checks", "TP", "TN",
        "FP", "FN", "coverage", "f1", "counts_scored", "counts_mae", "regions_scored",
        "regions_exact_set_match_rate"]].set_index(["dataset", "experiment", "situation", "group"]))

# Native DentVLM sub-experiments are summarized here; condition/task rows and image IDs are in overview/*.csv.
if not STAGE_COMPARISON.empty:
    stage_all = STAGE_COMPARISON[STAGE_COMPARISON["condition"] == "ALL"]
    print("===== saved stage transitions =====")
    display(stage_all[["dataset", "experiment", "comparison", "transition", "checks"]].set_index(
        ["dataset", "experiment", "comparison", "transition"]))
if not PHRASING_COMPARISON.empty:
    print("===== phrasing agreement =====")
    phrasing_summary = PHRASING_COMPARISON.groupby(["dataset", "experiment", "status"], as_index=False)[
        ["checks", "unresolved_phrasings"]].sum()
    display(phrasing_summary.set_index(["dataset", "experiment", "status"]))
if not VOTE_REPLAY_COMPARISON.empty:
    print("===== union vs majority replay of saved rationale regions =====")
    display(VOTE_REPLAY_COMPARISON[["dataset", "experiment", "region_vote", "TP", "TN", "FP",
        "FN", "coverage", "f1", "regions_scored", "regions_excluded",
        "regions_exact_set_match_rate", "regions_mean_jaccard"]].set_index(
        ["dataset", "experiment", "region_vote"]))
if not PARSE_RECOVERY_COMPARISON.empty:
    print("===== parse outcomes =====")
    recovery_summary = PARSE_RECOVERY_COMPARISON.groupby(
        ["dataset", "experiment", "stage", "status"], as_index=False)["checks"].sum()
    display(recovery_summary.set_index(["dataset", "experiment", "stage", "status"]))
if not CALL_USAGE_COMPARISON.empty:
    print("===== analyzer usage =====")
    display(CALL_USAGE_COMPARISON[["dataset", "experiment", "stage", "attempt", "calls",
        "prompt_tokens", "completion_tokens", "latency_seconds"]].set_index(
        ["dataset", "experiment", "stage", "attempt"]))

# Paired comparison against the first complete experiment: the same images and the same ground truth, so the
# columns say what a knob changed (checks corrected/worsened), not what the image sample was. Location is
# left out here because each experiment has its own location truth in the table above.
COMPARISONS = {}
for spec in DATASETS:
    dataset = spec["name"]
    complete = {c["name"]: xp.run_dir(c, dataset) for c in EXPERIMENTS
                if (c["name"], dataset) in REPORTS and not REPORTS[c["name"], dataset]["missing_results"]}
    if len(complete) < 2:
        continue
    COMPARISONS[dataset] = da.compare_runs(GT[dataset], complete, dataset=dataset, evaluate_location=False)
    ev.write_report(COMPARISONS[dataset], Path(OUTPUT_ROOT) / "comparison" / dataset)
    print(f"\n===== {dataset}: paired against {next(iter(complete))} =====")
    display(pd.DataFrame(COMPARISONS[dataset]["run_comparison"])[
        ["run", "model", "phrasings", "region_vote", "location", "count_question", "ask_untrained",
         "not_assessed_checks", "expected_finding_checks", "scored_finding_checks", "TP", "TN", "FP",
         "FN", "coverage", "sensitivity", "specificity", "ppv", "f1", "paired_checks",
         "paired_reference_f1", "paired_run_f1", "paired_f1_delta", "corrected", "worsened",
         "left_not_assessed", "became_not_assessed", "left_unresolved", "became_unresolved",
         "mean_calls_per_image", "mean_inference_calls_per_image", "cache_hits", "cache_hit_rate",
         "completion_tokens", "inference_completion_tokens"]].set_index("run"))
print(f"\nFull joined tables: {Path(OUTPUT_ROOT) / 'overview'}")
print("Coverage = scored / expected checks. Not-assessed findings have no active DentVLM task and are never "
      "counted as negatives. Unparseable answers are excluded from TP/FP/TN/FN and reported separately. "
      "Empty denominators stay N/A. Paired F1 uses only checks asked and resolved by both runs.")

In [ ]:
# ============================================================
# CELL 12 - Drill into one experiment without losing its DentVLM-specific diagnostics
# ============================================================
INSPECT_EXPERIMENT = EXPERIMENTS[0]["name"]  # any name from CELL 3
INSPECT_DATASET = DATASETS[0]["name"]

report = REPORTS.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if report is None:
    print(f"No evaluation for {INSPECT_EXPERIMENT}/{INSPECT_DATASET}; run CELLs 9 and 11 first.")
else:
    print(f"===== {INSPECT_EXPERIMENT} / {INSPECT_DATASET} =====")
    selected_overview = LEADERBOARD[(LEADERBOARD["experiment"] == INSPECT_EXPERIMENT) &
                                    (LEADERBOARD["dataset"] == INSPECT_DATASET)]
    display(selected_overview.set_index(["dataset", "experiment"]))
    selected_findings = FINDING_COMPARISON[(FINDING_COMPARISON["experiment"] == INSPECT_EXPERIMENT) &
                                           (FINDING_COMPARISON["dataset"] == INSPECT_DATASET)]
    if not selected_findings.empty:
        display(selected_findings[["condition", "assessment_status", "trained_task", "annotated_images",
            "annotated_positives", "not_assessed_checks", "scored_checks", "TP", "TN", "FP", "FN",
            "unparseable", "coverage", "sensitivity", "specificity", "ppv", "f1", "whole_image_f1",
            "count_n_scored", "count_exact_rate", "count_mae", "localized_cases", "location_f1",
            "region_exact_rate", "region_jaccard"]].set_index("condition"))
    if report["whole_image"]:
        print("whole-image answers alone (what the cells recovered, and what it cost):")
        display(pd.DataFrame(report["whole_image"]).set_index("condition"))
    if report["counts"]:
        display(pd.DataFrame(report["counts"]).set_index("condition"))
    if report["regions"]:
        display(pd.DataFrame(report["regions"]).set_index("condition"))
    if report["region_presence"]:
        print("presence per cell (every cell of every image, whatever the whole image said; rationale: named = present):")
        display(pd.DataFrame(report["region_presence"]).set_index(["condition", "region"]))
    # Full columns and per-finding transitions remain in JSON/CSV, with supporting image IDs.
    for table, title, columns in [
        ("stage_changes", "Saved decisions: first phrasing -> vote; whole image -> crops",
         ["comparison", "transition", "checks"]),
        ("phrasing_votes", "Agreement of saved phrasings (after any parse repairs)",
         ["task", "status", "checks", "unresolved_phrasings"]),
        ("region_vote_comparison", "Union vs majority replay on the same saved rationale answers",
         ["region_vote", "expected_finding_checks", "coverage", "regions_scored", "regions_excluded",
          "regions_exact_set_match_rate", "regions_mean_jaccard"]),
        ("parse_recovery", "Recorded question recovery (composite/extra tasks lack individual correctness labels)",
         ["stage", "task", "status", "checks", "correctness_scored", "correct", "correct_rate"]),
        ("call_usage", "Recorded analyzer completions and parse-retry usage",
         ["stage", "attempt", "calls", "prompt_tokens", "prompt_tokens_recorded_calls",
          "completion_tokens", "completion_tokens_recorded_calls", "latency_seconds", "latency_seconds_recorded_calls"]),
        ("case_breakdown", "Case breakdowns (named cells are location evidence, not tooth counts)",
         ["situation", "group", "not_assessed_checks", "expected_finding_checks", "coverage", "TP", "TN", "FP", "FN",
          "sensitivity", "specificity", "counts_scored", "counts_mae", "regions_scored", "regions_exact_set_match_rate"]),
    ]:
        rows = report.get(table, [])
        if table == "stage_changes":
            rows = [r for r in rows if r["condition"] == "ALL"]
        if rows:
            print(title)
            display(pd.DataFrame(rows)[columns])
    selected_situation_findings = SITUATION_FINDING_COMPARISON[
        (SITUATION_FINDING_COMPARISON["experiment"] == INSPECT_EXPERIMENT) &
        (SITUATION_FINDING_COMPARISON["dataset"] == INSPECT_DATASET)]
    if not selected_situation_findings.empty:
        print("Situation x finding detail (full rows and image IDs are saved in overview/):")
        display(selected_situation_findings[["situation", "group", "condition", "assessment_status",
            "annotated_images", "not_assessed_checks", "TP", "TN", "FP", "FN", "coverage", "f1",
            "count_n_scored", "count_mae", "localized_cases", "location_f1",
            "region_exact_rate"]].set_index(["situation", "group", "condition"]))
    if not report["parse_recovery"]:
        print("No saved question-attempt metadata for recovery analysis.")
    datasets = [r for (n, d), r in REPORTS.items() if n == INSPECT_EXPERIMENT]
    if len(datasets) > 1:
        print("\n===== pooled over the findings every dataset scores =====")
        pooled = ev.pooled_presence(datasets)
        if pooled:
            display(pd.DataFrame(pooled).set_index("condition"))
print("Case slices are descriptive. Empty denominators are N/A; coverage excludes not-assessed findings. "
      "Count/location scores use each run's true positives. Replay uses saved, repaired answers, not a retries-OFF run.")

In [ ]:
# ============================================================
# CELL 13 - Side convention check, then one image: dentist summary and raw answers
# ============================================================
# DentVLM's "left"/"right" follow Table S6 (its "left posterior" = FDI quadrants 1 and 4, the patient's right,
# which is the image left). Agreement far above 50% confirms dental_pipeline.LEFT_IS_IMAGE_LEFT. Far below means
# set LEFT_IS_IMAGE_LEFT = False in dental_pipeline.py, run `import importlib; importlib.reload(dp); importlib.reload(ev)`,
# and re-run CELL 11: no new model calls, the saved replies keep DentVLM's own words and the adapted truth is re-mapped.
INSPECT_IMAGE = None   # None = first image of the dataset
SHOW_RAW_CALLS = False

cfg = next(c for c in EXPERIMENTS if c["name"] == INSPECT_EXPERIMENT)
results = dp.load_results(xp.run_dir(cfg, INSPECT_DATASET))
if cfg["evaluate_location"]:
    print(INSPECT_DATASET, "side agreement:", ev.side_agreement(GT[INSPECT_DATASET], results))

image_id = INSPECT_IMAGE or next(iter(sorted(results)))
result = results[image_id]
print(f"\n{INSPECT_EXPERIMENT} / {INSPECT_DATASET} / {image_id}\n")
print(dp.dentist_report(result))
truth = [b["condition"] for b in GT[INSPECT_DATASET][image_id]["boxes"]]
print("\nGround-truth boxes:", {c: truth.count(c) for c in dict.fromkeys(truth)} or "none")
adapted = ADAPTED.get((INSPECT_EXPERIMENT, INSPECT_DATASET))
if adapted:
    for k, record in enumerate(adapted[image_id]["boxes"], 1):
        print(f"  box {k}: {record['condition']} -> {record['regions']} ({record['source']}; fixed windows {record['geometry']})")
if SHOW_RAW_CALLS:
    for call in result["calls"]:
        print(f"\n--- {call['stage']} | {call['task']} | {call['cell']} | finish={call['finish_reason']}")
        print(call["text"][:800])

In [ ]:
# ============================================================
# CELL 14 - Dentist report: one LLM call per image over the structured findings (resumable)
# ============================================================
# DentVLM answered 13 yes/no questions per image (39 with three phrasings, 91 with crops, plus optional counts) and
# named locations in its rationales. The report writer gets them as one dense JSON (every benchmark finding and extra
# task with its verbatim question and parsed answer, every region, the multiplicity, explicit statuses incl.
# not_assessed), returns a report in the experiment's report_language as JSON (one entry per finding in seven
# sections, impression, caveats), which is verified against the data (every finding exactly once, statuses unchanged,
# nothing invented), sent back once for correction if it fails, and rendered to Markdown. One .json + one .md per
# image under <name>/<dataset>/reports/<model>-<language>/reports; a reply that fails twice keeps the deterministic
# dentist summary, marked as such. The model never sees the image.
from IPython.display import Markdown, display

REPORT_EXPERIMENTS = [INSPECT_EXPERIMENT]  # one call per image per experiment; [c["name"] for c in EXPERIMENTS] for all

WRITTEN = {}
for cfg in [c for c in EXPERIMENTS if c["name"] in REPORT_EXPERIMENTS]:
    writer = xp.report_writer(cfg)
    print(f"{cfg['name']}: report writer {writer.public()}")
    for spec in DATASETS:
        name, dataset = cfg["name"], spec["name"]
        results = dp.load_results(xp.run_dir(cfg, dataset))
        if not results:
            print(f"{name}/{dataset}: no results; run CELL 9 first.")
            continue
        WRITTEN[name, dataset] = rw.report_dataset(
            writer, results, xp.run_dir(cfg, dataset) / "reports" / writer.run_name,
            analyzer=xp.analyzer_name(cfg), resume=True, limit=cfg["report_images"])
        print(f"{name}/{dataset}: {rw.summarize_reports(WRITTEN[name, dataset])}")

# One report to read (the image inspected in CELL 13 when it has one).
reports = WRITTEN.get((INSPECT_EXPERIMENT, INSPECT_DATASET)) or next(iter(WRITTEN.values()), {})
if reports:
    shown = reports.get(globals().get("image_id")) or reports[next(iter(sorted(reports)))]
    print(f"{shown['image_id']}: verified={shown['verified']} | attempts={len(shown['attempts'])} | "
          f"problems={shown['problems']}")
    display(Markdown(shown["markdown"]))